In [1]:
from pathlib import Path
import numpy as np
import tifffile

try:
    import tomllib as _toml          # Python 3.11+
except ModuleNotFoundError:          # older Python
    import tomli as _toml

In [2]:
def make_split_indices(n, mode="interleaved", seed=0):
    """
    Return (idx1, idx2): disjoint, sorted index arrays covering range(n).

    mode = "interleaved" : even indices -> half1, odd indices -> half2 (standard odd/even split;
                           both halves cover the full angular range)
    mode = "random"      : random 50/50 split (reproducible via `seed`)
    mode = "blocks"      : first half / second half of the series (NOT recommended for FSC,
                           since each half covers a different angular range)
    """
    idx = np.arange(n)
    if mode == "interleaved":
        return idx[0::2], idx[1::2]
    if mode == "random":
        rng = np.random.default_rng(seed)
        perm = rng.permutation(n)
        h = (n + 1) // 2
        return np.sort(perm[:h]), np.sort(perm[h:])
    if mode == "blocks":
        h = (n + 1) // 2
        return idx[:h], idx[h:]
    raise ValueError(f"Unknown split mode: {mode!r}")


def _fmt_scalar(v):
    if isinstance(v, bool):
        return "true" if v else "false"
    if isinstance(v, str):
        return '"' + v.replace("\\", "\\\\").replace('"', '\\"') + '"'
    if isinstance(v, int):
        return str(v)
    if isinstance(v, float):
        return repr(v)
    raise TypeError(f"Unsupported TOML value type: {type(v)}")


def _fmt_value(v, float_fmt="{:.8f}"):
    """Serialise scalars, flat lists and lists of lists (the types used in the alignment file)."""
    if isinstance(v, list):
        if v and isinstance(v[0], list):
            return "[" + ", ".join(_fmt_value(x, float_fmt) for x in v) + "]"
        items = [float_fmt.format(x) if isinstance(x, float) else _fmt_scalar(x) for x in v]
        return "[" + ", ".join(items) + "]"
    return _fmt_scalar(v)


def read_toml_with_header(path):
    """Return (parsed_dict, list_of_leading_comment_lines)."""
    text = Path(path).read_text()
    header = []
    for line in text.splitlines():
        if line.startswith("#") or not line.strip():
            if line.strip():
                header.append(line)
            continue
        break
    return _toml.loads(text), header


def write_toml(path, data, header):
    lines = list(header) + [""]
    for k, v in data.items():
        if isinstance(v, list) and v and isinstance(v[0], list):
            fmt = "{:.6f}"          # shifts_px style
        elif isinstance(v, list):
            fmt = "{:.8f}"          # angles style
        else:
            fmt = "{}"
        if isinstance(v, float):
            lines.append(f"{k} = {v!r}")
        else:
            lines.append(f"{k} = {_fmt_value(v, fmt)}")
    Path(path).write_text("\n".join(lines) + "\n")


def _out_path(src, tag, out_dir):
    src = Path(src)
    return Path(out_dir) / f"{src.stem}_{tag}{src.suffix}"


In [3]:
def split_dataset(tiff_path, angles_path, alignment_path,
                  out_dir=None, mode="interleaved", seed=0, save_indices=True):
    """
    Split a tilt series into two independent halves and save matching TIFF / angle / alignment files.

    Returns a dict with the output paths and the indices used for each half.
    """
    tiff_path, angles_path, alignment_path = map(Path, (tiff_path, angles_path, alignment_path))
    out_dir = Path(out_dir) if out_dir else tiff_path.parent
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- load -------------------------------------------------------------------------------
    stack = tifffile.imread(tiff_path)
    angles = np.loadtxt(angles_path, ndmin=1)
    align, header = read_toml_with_header(alignment_path)

    n = stack.shape[0]
    if stack.ndim != 3:
        raise ValueError(f"Expected a 3D TIFF stack (N, H, W), got shape {stack.shape}")
    if len(angles) != n:
        raise ValueError(f"Angle file has {len(angles)} entries but TIFF has {n} projections")

    per_proj_keys = [k for k, v in align.items() if isinstance(v, list) and len(v) == n]
    if not per_proj_keys:
        raise ValueError("No per-projection lists of length N found in the alignment file")

    idx1, idx2 = make_split_indices(n, mode=mode, seed=seed)
    print(f"{n} projections -> half1: {len(idx1)}, half2: {len(idx2)}  (mode={mode})")
    print(f"Per-projection alignment keys split: {per_proj_keys}")

    # ---- write ------------------------------------------------------------------------------
    outputs = {}
    for tag, idx in (("half1", idx1), ("half2", idx2)):
        tiff_out = _out_path(tiff_path, tag, out_dir)
        ang_out = _out_path(angles_path, tag, out_dir)
        aln_out = _out_path(alignment_path, tag, out_dir)

        tifffile.imwrite(tiff_out, stack[idx])                       # same dtype as input
        np.savetxt(ang_out, angles[idx], fmt="%.18e")                # same format as input

        sub = dict(align)
        for k in per_proj_keys:
            sub[k] = [align[k][i] for i in idx]
        if "label" in sub:
            sub["label"] = f"{sub['label']}_{tag}"
        write_toml(aln_out, sub, header)

        if save_indices:
            np.savetxt(out_dir / f"split_indices_{tag}.txt", idx, fmt="%d")

        outputs[tag] = dict(tiff=tiff_out, angles=ang_out, alignment=aln_out, indices=idx)
        print(f"[{tag}] {tiff_out.name}, {ang_out.name}, {aln_out.name}")

    return outputs

In [14]:
if __name__ == "__main__" or "get_ipython" in globals():
    TIFF_PATH      = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\105\projections\105_Gamma0_OD_ASYM_stack_COR_unaligned_2.tiff"
    ANGLES_PATH    = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\105\projections\105_Gamma0_angles_COR_unaligned_skipped_and_shifted.txt"
    ALIGNMENT_PATH = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\105\projections\Gamma0_alignment.toml"
    OUT_DIR        = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\105\projections\for_fsc"       # created if missing

    results = split_dataset(
        TIFF_PATH, ANGLES_PATH, ALIGNMENT_PATH,
        out_dir=OUT_DIR,
        mode="interleaved",   # "interleaved" (odd/even), "random", or "blocks"
        seed=0,               # only used for mode="random"
    )

65 projections -> half1: 33, half2: 32  (mode=interleaved)
Per-projection alignment keys split: ['corrected_angles_deg', 'shifts_px']
[half1] 105_Gamma0_OD_ASYM_stack_COR_unaligned_2_half1.tiff, 105_Gamma0_angles_COR_unaligned_skipped_and_shifted_half1.txt, Gamma0_alignment_half1.toml
[half2] 105_Gamma0_OD_ASYM_stack_COR_unaligned_2_half2.tiff, 105_Gamma0_angles_COR_unaligned_skipped_and_shifted_half2.txt, Gamma0_alignment_half2.toml


In [ ]:


# if __name__ == "__main__" or "get_ipython" in globals():
#     TIFF_PATH      = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\270\projections\270_Gamma45_OD_ASYM_stack_COR_unaligned_2.tiff"
#     ANGLES_PATH    = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\270\projections\270_Gamma45_angles_COR_unaligned_2_skipped_and_shifted.txt"
#     ALIGNMENT_PATH = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\270\projections\Gamma45_alignment.toml"
#     OUT_DIR        = r"C:\Users\emmab\Projects\Vector_Tomography\data_dir\270\projections\for_fsc"       # created if missing

#     results = split_dataset(
#         TIFF_PATH, ANGLES_PATH, ALIGNMENT_PATH,
#         out_dir=OUT_DIR,
#         mode="interleaved",   # "interleaved" (odd/even), "random", or "blocks"
#         seed=0,               # only used for mode="random"
#     )

9 projections -> half1: 5, half2: 4  (mode=interleaved)
Per-projection alignment keys split: ['corrected_angles_deg', 'shifts_px']
[half1] 270_Gamma45_OD_ASYM_stack_COR_unaligned_2_half1.tiff, 270_Gamma45_angles_COR_unaligned_2_skipped_and_shifted_half1.txt, Gamma45_alignment_half1.toml
[half2] 270_Gamma45_OD_ASYM_stack_COR_unaligned_2_half2.tiff, 270_Gamma45_angles_COR_unaligned_2_skipped_and_shifted_half2.txt, Gamma45_alignment_half2.toml
